# ML-03 — My Lane as an ML Task

*Phase: Foundations. Bridges the ML-systems map onto one concrete, buildable task.*

**Lane:** Refresh / Content Opportunity Scoring
**Builds on:** `w01_research_question.ipynb` (decision, action, cost of a wrong call)

> Skills loaded: `skills/framing-ml-problems/SKILL.md` + `skills/flyrank/flyrank-data/SKILL.md`.

**Core first, AI second.** I used an assistant to pressure-test this framing — to argue the opposite case and to check my reasoning against the pipeline source. The framing is stated in my own words, and every number in the code cells is printed at runtime, not typed in by me.

## 1. My lane as an ML task (type)

**Task type: ranking / scoring.**

The framing skill maps a question by its shape. Mine is *"which pages should a reviewer open **first**?"* — a "which ones first?" question, which is ranking: the target is a priority score, and the metric is precision@K.

Deliberately **not** the other three:

| Task type | Why not |
|---|---|
| Classification | A yes/no verdict on 30,000 pages isn't the decision. Nobody acts on "page #18,402 = declining"; they act on an ordered shortlist. |
| Clustering | No target. Useful later for *describing* page types, but it cannot say which to open first. |
| Signal analysis | Answers "which signals travel together?" — a question about the data, not a queue anyone can work from. |

### The wrinkle: ranking implemented through a classifier

The only label available is binary, so `03_train_model.py` trains classifiers. That is not a contradiction — a classifier emits a **probability**, and that probability *is* the priority score:

```
ranking question
   -> trained as binary classification (that is the label I have)
   -> model emits a probability per page
   -> probability IS the priority score
   -> pages sorted by it; top K becomes the review queue
   -> scored with Precision@K
```

I can see this in the shipped `outputs/model_report.md`: the queue preview carries both a `score` and a `model probability` per row, and the queue is ordered by it.

Two consequences worth stating now:

- **No probability threshold exists in this design.** The cut is *positional* (top K), not a value cut-off. Moving a 0.5 cut-off to 0.3 changes how many pages get *labelled* declining and changes nothing about which pages get opened — the ordering is untouched.
- **The knob for the w01 cost asymmetry is K, not a threshold.** False negatives are the expensive side (they compound silently and unobserved), so leaning toward catching more declines means **raising K**: more reviewer hours, fewer silent misses.

### Where this sits on the ML loop

```
data -> features -> [target: is_declining proxy] -> model -> probability
     -> rank -> top-K queue (+ reason codes) -> HUMAN REVIEWER
     -> refresh / expand / monitor
```

The loop ends at a person. Nothing in it publishes or edits anything — `04_evaluate_and_export.py` emits a queue, not an action.

## 2. Target or proxy

**The target is a proxy, and a rule-defined one. Naming that plainly is the point of this section.**

```python
is_declining_label = (trend_direction == "down")
```

and `trend_direction` is itself a threshold rule over `trend_pct`:

```
trend_pct = (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d * 100
down = trend_pct < -20%      up = trend_pct > +20%
new  = prev 0 & last > 0     flat = both 0        stable = everything else
```

So the target is two rules deep: a ratio, cut at −20%, then read as a category.

- **What I actually care about:** "this page is worth a reviewer's time."
- **What I can measure:** "this page's impressions fell more than 20% month-on-month."

The framing skill's rule is explicit: *the target must be observed, not defined — a label that comes from someone's rule means your model learns the rule, not the world.* My target does not pass that test. I'd rather write it here than have a reviewer find it.

### The gap runs in both directions

| Direction | Example | Consequence |
|---|---|---|
| Labelled declining, didn't need review | A page dipping seasonally, or one whose traffic a sibling page absorbed | Rewarded into the queue — reviewer time spent for nothing |
| Needed review, not labelled declining | A weak page flat near zero for a year: it never "fell" because it never rose | Invisible to the model **forever** |

The second row is the hard ceiling: those pages cannot be surfaced by *any* model trained on this label, however good. That is a limit of the framing, not a tuning problem.

The −20% cut has its own edge: a page at −19.9% and one at −20.1% get opposite labels while being effectively identical. Near the boundary, the label is close to arbitrary.

### Leakage guard

Because the label **is** `trend_direction`, neither `trend_direction` nor `trend_pct` may ever be a feature. A model given them would score near-perfectly and have learned nothing — it would be reading its own answer key. (The repo's notebook 02 demonstrates exactly this.) IDs (`content_id`, `client_id`) are pseudonyms: grouping and splitting only, never features.

**A subtler one I want on the record for Week 3.** The shipped model's top two features are `days_with_impressions` (0.158) and `log_impressions_90d` (0.128). Neither is banned — but `trend_direction` is computed from `impressions_last_30d` vs `impressions_prev_30d`, and *both of those windows sit inside the same 90-day totals*. A page that collapsed to zero mechanically has fewer active days and a smaller 90-day total. That is not leakage in the strict sense, but it is **adjacency by construction**, and it plausibly explains why those two features dominate. Flagging it now; testing it in `w03_feature_leakage_check.ipynb`.

### The cleaner target I am not using yet

An **observed outcome in a later window**: does this page's performance actually deteriorate over the *next* 90 days? That is measured, not defined, and it removes the circularity. The warehouse release (`fact_content_daily_performance`, ~79M rows, daily grain) can support two separate windows. Noting it as the capstone's most valuable upgrade — with the data-skill's warning attached: feature window and label window must be lined up first, or the fix becomes a worse leak than the problem.

## 3. Success metric

**Primary: Precision@50** — of the top 50 pages the model ranks, what share carry the declining label?

It measures the decision that is actually taken. Accuracy would grade 30,000 separate yes/no verdicts; the reviewer makes *one* decision — which K pages to open. Ranking quality below the fold is invisible to everybody.

**K = 50 is a placeholder for one review cycle's realistic capacity.** If the real capacity is 20 or 200, K changes and the evaluation is redone. K follows the reviewer, not the repo.

### Reported beside two numbers, always

Precision@K alone flatters. From `outputs/model_report.md`, on the shipped run:

| Reference point | Precision@50 |
|---|---:|
| **Base rate** (random draw of 50) | **0.542** |
| Hand rule (`baseline_rules`) | 0.240 |
| logistic_regression | 0.400 |
| decision_tree | 0.540 |
| **random_forest** (selected) | **0.740** |

Read honestly, the headline is not "0.740". It is **+0.198 over a random draw** — roughly **10 extra useful pages in a 50-page cycle**. Real, worth having, and a much smaller claim than "74% precision" sounds.

Note the decision_tree at 0.540: statistically indistinguishable from picking 50 pages at random. Reported without the base rate beside it, it would look like a working model.

### ROC-AUC: secondary, diagnosis only — not for choosing models

AUC asks: take one declining page and one healthy page at random — does the model score the declining one higher? Threshold-free and base-rate-independent, which makes it a good **failure reader**:

| AUC | Precision@50 | Diagnosis |
|---|---|---|
| high | high | working |
| high | **low** | model learned, but the *top* of the list is bad — fix the ranking head |
| low | low | didn't learn — go back to features |

The shipped numbers illustrate why it can't be the deciding metric: decision_tree posts AUC 0.742 against random_forest's 0.750 — nearly identical — while their Precision@50 differ by 0.20. AUC weights a #29,000-vs-#29,500 comparison the same as #3-vs-#700, and no human ever sees the former. So AUC diagnoses; it does not decide.

**Named before training, on purpose.** The skill's warning: *"good" defined after the fact always looks good.* Writing the metric down now is what stops me from switching later to whichever number looks nicest.

### The split it is computed on

Group-held-out by **client** (`client_holdout`, as the shipped report records), never 20% of rows. A random row split would put pages from the same site on both sides, and the model would score well by memorising the site rather than learning anything transferable. **The base rate must be recomputed on the held-out clients** — 0.542 is the whole-file figure, and the test-split figure can differ.

## 4. The unit of analysis, as a real dataframe

**One row = one content page (`content_id`), aggregated over its trailing 90-day window.**

Not one row per day, not per client, not per keyword. That fixes what the model can express: page-level characteristics only. A question like "is this client declining overall?" is a *different* unit and needs a different table.

The data skill's gotchas apply to everything below, so I check them in code rather than trusting my reading:

- Rate columns are **×100 percentages** — `ctr = 0.76` means 0.76%, not 76%.
- **`avg_position = 0` means "no data"**, not rank zero.
- `scroll_rate` and `ai_traffic_pct` **can exceed 100** (different measurement systems).
- **Missingness follows `content_type`** — a blind `fillna(0)` silently encodes content type into the features.

In [6]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

RAW = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(RAW)

n_rows, n_cols = df.shape
print(f"Loaded: {n_rows:,} rows x {n_cols} columns")
print("(data dictionary says 30,000 x 44 -- if this differs, stop and re-check)")

print("\n" + "=" * 66)
print("IS ONE ROW REALLY ONE PAGE?")
print("=" * 66)
n_pages = df["content_id"].nunique()
print(f"rows              : {n_rows:,}")
print(f"unique content_id : {n_pages:,}")
if n_pages == n_rows:
    print("-> confirmed: one row = one page, no duplicates.")
else:
    print(f"-> WARNING: {n_rows - n_pages:,} duplicate content_id rows.")
    print("   The stated unit of analysis is wrong. Investigate before modelling.")

sizes = df.groupby("client_id").size()
print(f"\nclients          : {df['client_id'].nunique()}")
print(f"pages per client : median {sizes.median():.0f}, min {sizes.min()}, max {sizes.max()}")
print(f"largest client is {sizes.max() / n_rows:.1%} of all rows "
      f"-> client-holdout splits will be lumpy; check base rate per split.")

Loaded: 30,000 rows x 44 columns
(data dictionary says 30,000 x 44 -- if this differs, stop and re-check)

IS ONE ROW REALLY ONE PAGE?
rows              : 30,000
unique content_id : 30,000
-> confirmed: one row = one page, no duplicates.

clients          : 32
pages per client : median 567, min 3, max 7008
largest client is 23.4% of all rows -> client-holdout splits will be lumpy; check base rate per split.


In [7]:
# --- The target, built exactly as the pipeline builds it ----------------
print("=" * 66)
print("SKETCHING THE TARGET (a rule-defined proxy)")
print("=" * 66)

print("trend_direction is itself a rule over trend_pct. Its full distribution:\n")
counts = df["trend_direction"].value_counts(dropna=False)
for value, count in counts.items():
    flag = "  <-- becomes label = 1" if str(value).lower() == "down" else ""
    print(f"   {str(value):<10} {count:>7,}  ({count/n_rows:>6.1%}){flag}")

df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
base_rate = df["is_declining_label"].mean()

print(f"\nis_declining_label = (trend_direction == 'down')")
print(f"positives : {int(df['is_declining_label'].sum()):,}")
print(f"negatives : {int((1 - df['is_declining_label']).sum()):,}")
print(f"BASE RATE : {base_rate:.3f}   ({base_rate:.1%})")
print("(data dictionary states 16,262 rows = 54.2%)")

SKETCHING THE TARGET (a rule-defined proxy)
trend_direction is itself a rule over trend_pct. Its full distribution:

   down        16,262  ( 54.2%)  <-- becomes label = 1
   stable       5,962  ( 19.9%)
   up           4,388  ( 14.6%)
   new          2,236  (  7.5%)
   flat         1,152  (  3.8%)

is_declining_label = (trend_direction == 'down')
positives : 16,262
negatives : 13,738
BASE RATE : 0.542   (54.2%)
(data dictionary states 16,262 rows = 54.2%)


In [8]:
# --- What the base rate does to my metric -------------------------------
print("=" * 66)
print("BASE RATE vs. THE DOCUMENTED RESULTS")
print("=" * 66)

reference = {
    "random draw of 50":        base_rate,
    "hand rule (baseline)":     0.240,
    "logistic_regression":      0.400,
    "decision_tree":            0.540,
    "random_forest (selected)": 0.740,
}

print(f"{'reference point':<26} {'P@50':>7}  {'vs random':>10}  {'extra pages/50':>15}")
print("-" * 66)
for name, value in reference.items():
    delta = value - base_rate
    extra = delta * 50
    print(f"{name:<26} {value:>7.3f}  {delta:>+10.3f}  {extra:>+15.1f}")

print("\nHonest headline: the selected model is worth about "
      f"{(0.740 - base_rate) * 50:+.0f} extra useful pages per 50-page cycle.")
print("NOT '74% precision' -- that number is mostly the base rate.")

if 0.240 < base_rate:
    print("\n*** FINDING: the hand rule scores BELOW a random draw. ***")
    print("    Not a bug. 02_baseline_score.py ranks by")
    print("      0.40*visibility + 0.30*freshness_risk")
    print("    + 0.25*position_opportunity + 0.05*depth_gap")
    print("    -- which contains NO decline term at all. The rule aims at")
    print("    'worth refreshing'; the label measures 'impressions fell >20%'.")
    print("    Its 0.240 is a direct measurement of the gap between the two.")

BASE RATE vs. THE DOCUMENTED RESULTS
reference point               P@50   vs random   extra pages/50
------------------------------------------------------------------
random draw of 50            0.542      +0.000             +0.0
hand rule (baseline)         0.240      -0.302            -15.1
logistic_regression          0.400      -0.142             -7.1
decision_tree                0.540      -0.002             -0.1
random_forest (selected)     0.740      +0.198             +9.9

Honest headline: the selected model is worth about +10 extra useful pages per 50-page cycle.
NOT '74% precision' -- that number is mostly the base rate.

*** FINDING: the hand rule scores BELOW a random draw. ***
    Not a bug. 02_baseline_score.py ranks by
      0.40*visibility + 0.30*freshness_risk
    + 0.25*position_opportunity + 0.05*depth_gap
    -- which contains NO decline term at all. The rule aims at
    'worth refreshing'; the label measures 'impressions fell >20%'.
    Its 0.240 is a direct mea

In [9]:
# --- Gotcha checks from the data skill, verified rather than trusted ----
print("=" * 66)
print("DATA GOTCHAS -- CHECKED, NOT ASSUMED")
print("=" * 66)

print("1) Rate columns are x100 percentages, not fractions:")
for col in ["ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct"]:
    if col in df.columns:
        s = df[col].dropna()
        over_100 = int((s > 100).sum())
        note = f"  ({over_100:,} rows > 100 -- expected for this column)" if over_100 else ""
        print(f"   {col:<18} median {s.median():>8.2f}  max {s.max():>10.2f}{note}")

print("\n2) avg_position == 0 means NO DATA, not rank zero:")
if "avg_position" in df.columns:
    zero_pos = int((df["avg_position"] == 0).sum())
    print(f"   {zero_pos:,} rows ({zero_pos/n_rows:.1%}) -- dictionary says 1,205")
    print("   -> must be treated as missing, never as 'the best possible rank'")

print("\n3) Missingness is systematic, and follows content_type:")
if "content_type" in df.columns and "word_count" in df.columns:
    miss = (df.assign(wc_missing=df["word_count"].isna())
              .groupby("content_type")["wc_missing"].agg(["mean", "size"]))
    for ctype, row in miss.iterrows():
        print(f"   {str(ctype):<22} word_count missing in {row['mean']:>6.1%} "
              f"of {int(row['size']):,} rows")
    print("   -> a blind fillna(0) would encode content_type into the features")

DATA GOTCHAS -- CHECKED, NOT ASSUMED
1) Rate columns are x100 percentages, not fractions:
   ctr                median     0.07  max     100.00
   engagement_rate    median     0.00  max     100.00
   scroll_rate        median     5.00  max     300.00  (119 rows > 100 -- expected for this column)
   ai_traffic_pct     median     0.00  max     300.00  (23 rows > 100 -- expected for this column)

2) avg_position == 0 means NO DATA, not rank zero:
   1,205 rows (4.0%) -- dictionary says 1,205
   -> must be treated as missing, never as 'the best possible rank'

3) Missingness is systematic, and follows content_type:
   comparison article     word_count missing in   0.0% of 697 rows
   feedly article         word_count missing in   0.0% of 2,096 rows
   keyword article        word_count missing in  28.3% of 27,207 rows
   -> a blind fillna(0) would encode content_type into the features


In [10]:
# --- The unit of analysis, shown as an actual dataframe -----------------
print("=" * 66)
print("ONE ROW = ONE PAGE")
print("=" * 66)

show_cols = [c for c in [
    "content_id", "client_id", "content_type",
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "days_with_impressions", "word_count", "days_since_last_update",
    "is_declining_label",
] if c in df.columns]

display(df[show_cols].head(5))

print("\nRead one row out loud: this is ONE page, summarised over its trailing")
print("90-day window -- not a day, not a keyword, not a client.")
print("The last column is the target; everything before it is candidate context.")

ONE ROW = ONE PAGE


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,days_with_impressions,word_count,days_since_last_update,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,88,3221.0,20,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,88,2481.0,25,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,88,3515.0,20,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,88,NaN,22,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,88,2803.0,14,1



Read one row out loud: this is ONE page, summarised over its trailing
90-day window -- not a day, not a keyword, not a client.
The last column is the target; everything before it is candidate context.


In [11]:
# --- Which columns are actually allowed to be features ------------------
print("=" * 66)
print("FEATURE ELIGIBILITY (per scripts/ml_utils.py)")
print("=" * 66)

BANNED_LABEL_SOURCE = ["trend_direction", "trend_pct"]
BANNED_IDS = ["content_id", "client_id"]

print("NEVER features:")
for c in BANNED_LABEL_SOURCE:
    if c in df.columns:
        print(f"   {c:<22} label is derived from it -> leakage")
for c in BANNED_IDS:
    if c in df.columns:
        print(f"   {c:<22} pseudonym -> grouping/splitting only")

MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d",
    "log_ai_sessions_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

present = [c for c in MODEL_NUMERIC_FEATURES if c in df.columns]
derived = [c for c in MODEL_NUMERIC_FEATURES if c not in df.columns]

print(f"\nnumeric features used by the pipeline    : {len(MODEL_NUMERIC_FEATURES)}")
print(f"   already in the raw CSV                : {len(present)}")
print(f"   created by 01_prepare_features.py     : {len(derived)} -> {derived}")
print(f"categorical features                     : {len(MODEL_CATEGORICAL_FEATURES)}")

print("\nFLAG FOR WEEK 3 (leakage notebook):")
print("   The shipped model's top two features are days_with_impressions (0.158)")
print("   and log_impressions_90d (0.128). Both are permitted -- but the label")
print("   comes from impressions_last_30d vs impressions_prev_30d, and BOTH of")
print("   those windows sit inside the same 90-day totals. Adjacency by")
print("   construction, not strict leakage. To be tested, not assumed.")

FEATURE ELIGIBILITY (per scripts/ml_utils.py)
NEVER features:
   trend_direction        label is derived from it -> leakage
   trend_pct              label is derived from it -> leakage
   content_id             pseudonym -> grouping/splitting only
   client_id              pseudonym -> grouping/splitting only

numeric features used by the pipeline    : 18
   already in the raw CSV                : 14
   created by 01_prepare_features.py     : 4 -> ['log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d']
categorical features                     : 8

FLAG FOR WEEK 3 (leakage notebook):
   The shipped model's top two features are days_with_impressions (0.158)
   and log_impressions_90d (0.128). Both are permitted -- but the label
   comes from impressions_last_30d vs impressions_prev_30d, and BOTH of
   those windows sit inside the same 90-day totals. Adjacency by
   construction, not strict leakage. To be tested, not assumed.


In [12]:
# --- Does the pipeline's prep filter change anything on this slice? -----
print("=" * 66)
print("PREP FILTER -- IS MY BASE RATE THE SAME ONE THE PIPELINE USES?")
print("=" * 66)

# scripts/01_prepare_features.py keeps rows where:
#   impressions_90d > 0  AND  content_age_days >= 90,  then dedupes on content_id
kept = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
kept = kept.drop_duplicates(subset=["content_id"])

print(f"raw rows      : {n_rows:,}")
print(f"after filter  : {len(kept):,}")
print(f"dropped       : {n_rows - len(kept):,}")

prepared_rate = kept["is_declining_label"].mean()
print(f"\nbase rate, raw      : {base_rate:.4f}")
print(f"base rate, prepared : {prepared_rate:.4f}")

if len(kept) == n_rows:
    print("\n-> The filter is a NO-OP on this slice, exactly as the dictionary implies:")
    print("   every row already has impressions_90d >= 1 and content_age_days >= 90,")
    print("   and content_id is unique. So 0.542 is both the raw AND the prepared")
    print("   base rate -- the number I compare every Precision@50 against.")
else:
    print("\n-> The filter DOES drop rows here. Use the prepared base rate above,")
    print("   not the raw one, when reading Precision@50.")

PREP FILTER -- IS MY BASE RATE THE SAME ONE THE PIPELINE USES?
raw rows      : 30,000
after filter  : 30,000
dropped       : 0

base rate, raw      : 0.5421
base rate, prepared : 0.5421

-> The filter is a NO-OP on this slice, exactly as the dictionary implies:
   every row already has impressions_90d >= 1 and content_age_days >= 90,
   and content_id is unique. So 0.542 is both the raw AND the prepared
   base rate -- the number I compare every Precision@50 against.


### Section 3's open claim, now checked in code

§3 states that the base rate **must be recomputed on the held-out clients**, because 0.542 is
the whole-file figure. Below I build a client-grouped split of the same shape the pipeline uses
(~20% of clients held out) and read the base rate on each side.

This is an *illustration of the split's shape*, not a reproduction of the pipeline's exact
split — `make_client_aware_split` in `scripts/03_train_model.py` owns the real one. The point
is the spread, not the specific seed.

In [13]:
# --- Base rate is NOT one number: it moves with the client split --------
from sklearn.model_selection import GroupShuffleSplit

print("=" * 66)
print("BASE RATE ON A CLIENT-HELD-OUT SPLIT")
print("=" * 66)

groups = df["client_id"]
y = df["is_declining_label"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(df, y, groups))

print(f"whole file                  : base rate {y.mean():.3f}  ({len(df):,} rows)")
print(f"train ({df.iloc[train_idx]['client_id'].nunique()} clients)"
      f"{'':<11} : base rate {y.iloc[train_idx].mean():.3f}  ({len(train_idx):,} rows)")
print(f"holdout ({df.iloc[test_idx]['client_id'].nunique()} clients)"
      f"{'':<9} : base rate {y.iloc[test_idx].mean():.3f}  ({len(test_idx):,} rows)")

# How much does the holdout base rate move if I only change the seed?
rates = []
for seed in range(20):
    sp = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    _, t = next(sp.split(df, y, groups))
    rates.append(y.iloc[t].mean())

print(f"\nacross 20 different client splits, the holdout base rate ranges")
print(f"   {min(rates):.3f} to {max(rates):.3f}  (spread {max(rates)-min(rates):.3f})")
print("\n-> The 'random draw' reference is NOT a fixed 0.542. It shifts with which")
print("   clients land in the holdout, and that spread is on the same scale as the")
print("   model's advantage. Any Precision@50 must be quoted beside the base rate")
print("   OF ITS OWN SPLIT -- not the whole-file one.")

BASE RATE ON A CLIENT-HELD-OUT SPLIT
whole file                  : base rate 0.542  (30,000 rows)
train (25 clients)            : base rate 0.550  (23,837 rows)
holdout (7 clients)          : base rate 0.511  (6,163 rows)

across 20 different client splits, the holdout base rate ranges
   0.384 to 0.735  (spread 0.351)

-> The 'random draw' reference is NOT a fixed 0.542. It shifts with which
   clients land in the holdout, and that spread is on the same scale as the
   model's advantage. Any Precision@50 must be quoted beside the base rate
   OF ITS OWN SPLIT -- not the whole-file one.


### The other open claim: is 0.540 really "indistinguishable from random"?

§3 says the decision tree's 0.540 is statistically indistinguishable from a random draw of 50.
That is an empirical claim, so it should be measured rather than asserted. Below I draw many
random 50-page samples and look at how much Precision@50 moves by chance alone.

In [14]:
# --- How noisy is Precision@50 at K=50? --------------------------------
rng = np.random.default_rng(0)

print("=" * 66)
print("HOW MUCH DOES PRECISION@50 MOVE BY CHANCE ALONE?")
print("=" * 66)

labels = df["is_declining_label"].to_numpy()
draws = np.array([rng.choice(labels, size=50, replace=False).mean() for _ in range(5000)])

lo, hi = np.percentile(draws, [2.5, 97.5])
print(f"5,000 random draws of 50 pages:")
print(f"   mean       {draws.mean():.3f}   (= the base rate, as expected)")
print(f"   std dev    {draws.std():.3f}")
print(f"   95% range  {lo:.3f} to {hi:.3f}")

print("\nWhere the documented results fall in that range:")
for name, value in [("hand rule", 0.240), ("logistic_regression", 0.400),
                    ("decision_tree", 0.540), ("random_forest", 0.740)]:
    inside = lo <= value <= hi
    verdict = "INSIDE the random range -- not distinguishable from chance" if inside \
              else "outside the random range"
    print(f"   {name:<22} {value:.3f}   {verdict}")

print("\n-> This is why K matters as much as the model. At K=50 a swing of about")
print(f"   +/-{(hi-lo)/2:.3f} is pure luck, so the third decimal of 0.740 is not a")
print("   claim I can defend -- only the gap over the base rate is. It is also the")
print("   reason GUIDE.md reports the forest as 0.68-0.74 across library versions.")

HOW MUCH DOES PRECISION@50 MOVE BY CHANCE ALONE?
5,000 random draws of 50 pages:
   mean       0.542   (= the base rate, as expected)
   std dev    0.070
   95% range  0.400 to 0.680

Where the documented results fall in that range:
   hand rule              0.240   outside the random range
   logistic_regression    0.400   INSIDE the random range -- not distinguishable from chance
   decision_tree          0.540   INSIDE the random range -- not distinguishable from chance
   random_forest          0.740   outside the random range

-> This is why K matters as much as the model. At K=50 a swing of about
   +/-0.140 is pure luck, so the third decimal of 0.740 is not a
   claim I can defend -- only the gap over the base rate is. It is also the
   reason GUIDE.md reports the forest as 0.68-0.74 across library versions.


**One caveat on the cell above, so I don't overstate it.** This simulation samples 50 pages
from the *whole file*, while the reported Precision@50 figures come from a *client-held-out test
set*. The two are not strictly the same experiment, so the interval is an order-of-magnitude
guide to the noise, not a formal significance test on the shipped numbers.

It is enough to settle the question I asked, though: at K=50 the sampling noise is large enough
that 0.540 sits inside the range a random draw produces, and 0.740 does not. That is the honest
version of "the model beats the baseline" — a gap that survives the noise, not a decimal that
doesn't.

## 5. Why ML beats a fixed rule here

I don't have to *assume* a hand rule is insufficient — the rule exists, and it is measured.

| Approach | Precision@50 | Where it lives |
|---|---:|---|
| Random draw | 0.542 | — |
| Hand rule | 0.240 | `scripts/02_baseline_score.py` |
| Selected model | 0.740 | `scripts/03_train_model.py` |

**The framing skill's test:** ML earns its place only when the pattern is real but too messy to write by hand. Here it clears that bar — but the honest margin is **+0.198 over random**, about **10 extra useful pages per 50-page cycle**, not the +0.5 the hand-rule comparison suggests.

### The finding underneath the table

The hand rule scoring *below* a random draw is not incompetence. Reading `02_baseline_score.py`, its score is:

```
0.40 * visibility  +  0.30 * freshness_risk
+ 0.25 * position_opportunity  +  0.05 * depth_gap
```

There is **no decline term in it at all**. The rule ranks pages that are *visible, stale, mid-position and thin* — a defensible definition of "worth refreshing". The label asks a different question: "did impressions fall more than 20% month-on-month?"

So 0.240 is not "the rule is bad". It is a **direct measurement of the distance between the proxy and the goal** — and it is the single most useful number in this notebook, because it puts a figure on the gap I described in §2.

It also means the comparison "model beats rule 3x" is not apples-to-apples: the model is being graded on the target it was trained on, and the rule is being graded on a target it was never aiming at. **The fair comparison is model vs. random draw (+0.198).** I'll say it that way in the report.

### The counterfactual

If the model had landed near 0.542, the hand rule would be the better answer — cheaper, transparent, no retraining, no drift, and self-explaining. "ML won" is a measured outcome here, not an assumption I started from.

### I keep the rule anyway

The baseline emits human-readable reason codes — `declining_with_demand`, `low_ctr_visible_page`, `stale_visible_page`, `thin_visible_page`, `page_one_decay_risk`, `low_engagement_visible_page` — which a probability cannot. `04_evaluate_and_export.py` blends them: **the model supplies the ordering, the rule supplies the explanation the reviewer reads.** Dropping the baseline would produce a better-ranked queue that nobody trusts or can act on.

### Tied to a real content action

Each queued page carries a suggested action derived from its reason codes: `expand_and_refresh`, `refresh_and_review_ctr`, `refresh`, or `monitor`. A FlyRank content reviewer opens the top K and decides per page — refresh, expand, or leave. The score never publishes or edits anything; the reason codes are what make the call reviewable.

### The one-paragraph frame

> For a **FlyRank content reviewer**, deciding **which pages to open first for a possible refresh this cycle**, we will build a **ranked review queue with scores and reason codes** from the **anonymized 90-day content-performance slice (30,000 pages × 44 columns, 32 clients)**, scoring **each page's predicted probability of carrying the `is_declining_label` proxy**, measured by **Precision@50 reported beside the base rate (0.542) and the hand-rule baseline (0.240), on a client-held-out split**. A wrong call costs **~10–15 minutes of reviewer time in one direction, and weeks of silent compounding decline in the other**. A plain rule isn't enough because **the learned model adds ~0.198 over a random draw — about 10 extra useful pages per cycle — from signal spread across tangled features**. We will claim only **observed / directional / decision-support** results.

### What I can't claim, carried forward from w01

Every number here measures agreement with a **proxy**, not with truth. A model matching the proxy perfectly would inherit all of the proxy's mistakes. No causal claim is available — no experiment, no randomised holdout of refreshed vs. non-refreshed pages — and nothing here predicts or reverse-engineers Google's ranking.

## 6. Self-check

- [ ] **Task type named** — ranking/scoring, with the classifier-probability implementation explained (§1)
- [ ] **Target/proxy named** — rule-defined, two rules deep, with the gap described in both directions (§2)
- [ ] **Success metric named before training** — Precision@50 beside base rate and baseline, on a client-held-out split; AUC diagnosis-only (§3)
- [ ] **Unit of analysis shown as a real dataframe** — and verified in code that one row really is one page (§4)
- [ ] **Target column sketched in code**, base rate printed and matched against the dictionary (§4)
- [ ] **Data gotchas checked in code**, not just quoted: ×100 rates, `avg_position = 0`, systematic missingness (§4)
- [ ] **Prep filter checked** — confirmed a no-op on this slice, so the raw and prepared base rates are the same number (§4)
- [ ] **Base rate recomputed on a client holdout** — shown to move with the split, so it is quoted per-split, not as a fixed 0.542 (§4)
- [ ] **Precision@50 noise measured** — 0.540 falls inside the random-draw range, 0.740 does not; only the gap over base rate is claimed (§4)
- [ ] **Why ML over a fixed rule** — argued from measured numbers, with the honest margin (+0.198 over random) rather than the flattering one (§5)
- [ ] **Output tied to a real content action** — reviewer opens top K, decides refresh/expand/monitor (§5)
- [ ] **Leakage guard stated** — `trend_direction`/`trend_pct` never features; IDs for grouping only; adjacency-by-construction flagged for Week 3 (§2, §4)
- [ ] **Notebook executed top to bottom** (`Run All`), outputs saved in the committed `.ipynb`
- [ ] No client names, URLs, or private queries anywhere
- [ ] Careful words throughout: observed, measured, directional, decision-support
- [ ] Committed to `work/notebooks/w02_ml_task_framing.ipynb` — then submit the repo URL on the card